In [ ]:
import os, sys
try:
    !pip install simpy==4.0.1 openpyxl >/dev/null 2>&1
except Exception as e:
    print("pip install warning:", e)
try:
    !apt-get update -qq
    !apt-get install -y -qq ffmpeg imagemagick
except Exception as e:
    print("apt-get install warning:", e)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fonts-droid-fallback.
(Reading database ... 126675 files and directories currently installed.)
Preparing to unpack .../00-fonts-droid-fallback_1%3a6.0.1r16-1.1build1_all.deb ...
Unpacking fonts-droid-fallback (1:6.0.1r16-1.1build1) ...
Selecting previously unselected package libfftw3-double3:amd64.
Preparing to unpack .../01-libfftw3-double3_3.3.8-2ubuntu8_amd64.deb ...
Unpacking libfftw3-double3:amd64 (3.3.8-2ubuntu8) ...
Selecting previously unselected package liblqr-1-0:amd64.
Preparing to unpack .../02-liblqr-1-0_0.4.2-2.1_amd64.deb ...
Unpacking liblqr-1-0:amd64 (0.4.2-2.1) ...
Selecting previously unselected package imagemagick-6-common.
Preparing to unpack .../03-imagemagick-6-common_8%3a6.9.11.60+dfsg-1.3ubuntu0.22.04.5_all.deb ...
Unpacking imagemagick

In [ ]:
import simpy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle
from matplotlib.animation import FFMpegWriter, PillowWriter
from IPython.display import HTML, display
import random, warnings
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import matplotlib.image as mpimg
import copy, math
warnings.filterwarnings("ignore")

In [ ]:
# Parameters and Import Dataset
DATA_CSV = '/content/HASIL_DATA_PENGAMATAN_DES.csv'
DEFAULT_SERVERS = 5
DEFAULT_QUEUE_CAPACITY = 10   # None => infinite
SIM_TIME_PADDING = 60         # menit padding
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

In [ ]:
# Load Dataset
if os.path.exists(DATA_CSV):
    df = pd.read_csv(DATA_CSV)
    print("Loaded CSV:", DATA_CSV)
else:
    raise FileNotFoundError(f"Data file not found. Upload DATA_CSV to Colab.")

print("Columns:", df.columns.tolist())

# Ensure datetimes
for col in ['ArrivalTime','ServiceStartTime','ServiceEndTime']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Sort & check
if 'ArrivalTime' in df.columns:
    df = df.sort_values('ArrivalTime').reset_index(drop=True)
else:
    raise ValueError("Dataset must contain 'ArrivalTime' column")

# Compute interarrival & service durations (minutes)
df['InterArrival_min'] = df['ArrivalTime'].diff().dt.total_seconds().div(60).fillna(0)
if 'ServiceStartTime' in df.columns and 'ServiceEndTime' in df.columns:
    df['Service_min'] = (df['ServiceEndTime'] - df['ServiceStartTime']).dt.total_seconds().div(60)
elif 'ServiceDuration' in df.columns:
    df['Service_min'] = df['ServiceDuration']
else:
    raise ValueError("Dataset must contain 'ServiceStartTime' & 'ServiceEndTime' or 'ServiceDuration'")

# Clean
df = df[df['Service_min'] > 0].copy()
df = df[df['InterArrival_min'] >= 0].copy()

n_customers = len(df)
print(f"Observations: {n_customers}")
mean_interarrival = df.loc[df['InterArrival_min']>0,'InterArrival_min'].mean()
mean_service = df['Service_min'].mean()
arrival_rate = (1.0/mean_interarrival) if (mean_interarrival>0) else 0.0
service_rate = (1.0/mean_service) if (mean_service>0) else 0.0
print(f"Estimated mean interarrival = {mean_interarrival:.3f} min -> arrival_rate = {arrival_rate:.6f} /min")
print(f"Estimated mean service      = {mean_service:.3f} min -> service_rate = {service_rate:.6f} /min")


Loaded CSV: /content/HASIL_DATA_PENGAMATAN_DES.csv
Columns: ['Customer', 'ArrivalTime', 'ServiceStartTime', 'ServiceEndTime', 'Server']
Observations: 161
Estimated mean interarrival = 0.533 min -> arrival_rate = 1.875383 /min
Estimated mean service      = 1.864 min -> service_rate = 0.536339 /min


In [ ]:
# Simulation engine (multi-server, per-server queues)
class MultiServerSimulator:
    def __init__(self, env, num_servers=5, queue_capacity=None):
        self.env = env
        self.num_servers = num_servers
        self.servers = [simpy.Resource(env, capacity=1) for _ in range(num_servers)]
        self.queue_capacity = queue_capacity
        self.wait_times = []
        self.system_times = []
        self.busy_time = [0.0]*num_servers
        self.customers_completed = 0

def choose_server_shortest(sim: MultiServerSimulator):
    loads = []
    for s in sim.servers:
        load = len(s.queue) + (1 if len(s.users)>0 else 0)
        loads.append(load)
    return int(np.argmin(loads))

def customer_process_multiserver(env, name, service_time, sim: MultiServerSimulator,
                                 events=None, assigned_server_idx=None):
    """
    assigned_server_idx: can be None, or 1-based server index from dataset (we normalize to 0-based).
    This function records 'arrive' event using the assigned server (0-based) when available,
    so the animator can place the customer in the correct queue from arrival.
    """
    # normalize assigned server to 0-based int or None
    if assigned_server_idx is None:
        assigned_srv0 = None
    else:
        try:
            assigned_srv0 = int(assigned_server_idx) - 1
        except Exception:
            assigned_srv0 = None

    arrive = env.now
    # balk check (global)
    if sim.queue_capacity is not None:
        total_load = sum(len(s.users) + len(s.queue) for s in sim.servers)
        if total_load >= sim.queue_capacity + sim.num_servers:
            if events is not None:
                events.append(('balk', env.now, name, None))
            return

    if events is not None:
        # record arrival and include server index if we know it from dataset (0-based)
        events.append(('arrive', env.now, name, assigned_srv0))

    # choose server
    if assigned_srv0 is None:
        srv_idx = choose_server_shortest(sim)
    else:
        srv_idx = assigned_srv0
        # safety check
        if srv_idx < 0 or srv_idx >= sim.num_servers:
            srv_idx = choose_server_shortest(sim)

    server_res = sim.servers[srv_idx]
    req = server_res.request()
    yield req

    # start service
    wait = env.now - arrive
    sim.wait_times.append(wait)
    if events is not None:
        events.append(('start_service', env.now, name, srv_idx))

    yield env.timeout(service_time)
    service_end = env.now
    sim.system_times.append(service_end - arrive)
    sim.busy_time[srv_idx] += service_time
    sim.customers_completed += 1

    # release and record end
    server_res.release(req)
    if events is not None:
        events.append(('end_service', env.now, name, srv_idx))

In [ ]:
# Runner functions
def run_replay_with_server_reallabels(df, num_servers=5, queue_capacity=None, visualize=True):
    env = simpy.Environment()
    sim = MultiServerSimulator(env, num_servers=num_servers, queue_capacity=queue_capacity)
    events = [] if visualize else None
    first = df['ArrivalTime'].iloc[0]

    for idx, row in df.iterrows():
        arr_min = (row['ArrivalTime'] - first).total_seconds()/60.0
        service = float(row['Service_min'])
        assigned_raw = row.get('Server', None)
        if pd.isna(assigned_raw):
            assigned = None
        else:
            try:
                assigned = int(assigned_raw)   # keep 1-based from dataset
            except Exception:
                assigned = None

        # name as C{1-based}
        cust_name = f"C{idx+1}"

        def spawn(a, name, s, assigned_srv):
            def proc():
                yield env.timeout(max(0.0, a - env.now))
                yield env.process(customer_process_multiserver(env, name, s, sim, events, assigned_srv))
            return proc

        env.process(spawn(arr_min, cust_name, service, assigned)())

    # run until last observed end + padding (safe)
    last_end = (df['ServiceEndTime'].max() - first).total_seconds()/60.0 if 'ServiceEndTime' in df.columns else None
    until = (last_end + SIM_TIME_PADDING) if last_end is not None else (df['InterArrival_min'].sum() + SIM_TIME_PADDING)
    env.run(until=until)

    results = {
        'Average Utilization (%)': (sum(sim.busy_time) / (sim.num_servers * env.now))*100 if env.now>0 else 0.0,
        'Average Time in System (minutes)': np.mean(sim.system_times) if sim.system_times else 0.0,
        'Average Waiting Time (minutes)': np.mean(sim.wait_times) if sim.wait_times else 0.0,
        'Total Work Item Completed': sim.customers_completed,
        'SimTime_min': env.now
    }
    return results, events

def run_stochastic_multiserver(num_servers=5, arrival_rate_per_min=0.01, service_rate_per_min=0.02,
                              n_customers=None, queue_capacity=None, seed=42, visualize=False, max_run_time=1e7):
    random.seed(seed); np.random.seed(seed)
    env = simpy.Environment()
    sim = MultiServerSimulator(env, num_servers=num_servers, queue_capacity=queue_capacity)
    events = [] if visualize else None

    def arrival_gen(env, sim, arrival_rate, service_rate, n_customers):
        i = 0
        while True:
            if n_customers is not None and i >= n_customers:
                break
            inter = np.random.exponential(1.0/arrival_rate) if arrival_rate>0 else 1.0
            yield env.timeout(inter)
            i += 1
            st = np.random.exponential(1.0/service_rate) if service_rate>0 else 1.0
            env.process(customer_process_multiserver(env, f"C{i}", st, sim, events, assigned_server_idx=None))

    env.process(arrival_gen(env, sim, arrival_rate_per_min, service_rate_per_min, n_customers))
    env.run()
    if env.now > max_run_time:
        print("Warning: reached max_run_time:", env.now)

    results = {
        'Average Utilization (%)': (sum(sim.busy_time) / (sim.num_servers * env.now))*100 if env.now>0 else 0.0,
        'Average Time in System (minutes)': np.mean(sim.system_times) if sim.system_times else 0.0,
        'Average Waiting Time (minutes)': np.mean(sim.wait_times) if sim.wait_times else 0.0,
        'Total Work Item Completed': sim.customers_completed,
        'SimTime_min': env.now
    }
    return results, events

In [ ]:
# Animation
def create_animation_per_server(events, num_servers, title="Simulation", save_path=None, fps=8, avatar_map=None):
    if not events or len(events) == 0:
        print("⚠️ No events to animate.")
        return None

    # Sort and time
    events = sorted(events, key=lambda x: x[1])
    times = [e[1] for e in events]
    tmin, tmax = min(times), max(times)
    duration = max(1.0, tmax - tmin)
    nframes = int(duration * fps) + 1
    frame_times = np.linspace(tmin, tmax, nframes)

    # Cashier and queue position
    server_xs = np.linspace(0.1, 0.9, num_servers)
    y_service = 0.8
    gap = 0.05
    max_q = 10
    q_positions = [[(sx, y_service - (j+1)*gap) for j in range(max_q)] for sx in server_xs]
    service_pos = [(sx, y_service) for sx in server_xs]

    server_q = [[] for _ in range(num_servers)]
    in_service = [None]*num_servers
    cust_pos = {}
    cust_state = {}

    # Build snapshot everytime
    frame_list = []
    events_copy = events.copy()
    for t in frame_times:
        while events_copy and events_copy[0][1] <= t:
            typ, et, name, sidx = events_copy.pop(0)
            s = 0 if sidx is None else int(max(0, min(num_servers-1, sidx)))
            if typ == "arrive":
                server_q[s].append(name)
                cust_state[name] = f"queued_{s}"
            elif typ == "start_service":
                if name in server_q[s]:
                    server_q[s].remove(name)
                in_service[s] = name
                cust_state[name] = f"in_service_{s}"
            elif typ == "end_service":
                in_service[s] = None
                cust_state[name] = "done"

        # Update visual position
        for s in range(num_servers):
            for j, nm in enumerate(server_q[s]):
                if j < len(q_positions[s]):
                    cust_pos[nm] = q_positions[s][j]
            if in_service[s] is not None:
                cust_pos[in_service[s]] = service_pos[s]

        frame_list.append([(n, *cust_pos[n], cust_state.get(n, "")) for n in cust_pos if cust_state.get(n) != "done"])

    # Setup figure
    fig, ax = plt.subplots(figsize=(10,6))
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.axis("off")
    ax.set_title(title, fontsize=13)

    # Draw box cashier
    for i, (sx, sy) in enumerate(service_pos):
        ax.add_patch(Rectangle((sx-0.03, sy-0.03), 0.06, 0.06, fill=False, lw=1))
        ax.text(sx, sy+0.05, f"Kasir {i+1}", ha="center", fontsize=9)

    time_text = ax.text(0.8, 0.95, "", fontsize=10, bbox=dict(facecolor="white", alpha=0.7))
    drawn = {}

    def draw_customer(name, x, y, state):
        color = "orange" if "queue" in state else ("green" if "service" in state else "gray")
        circ = plt.Circle((x, y), 0.018, color=color)
        ax.add_patch(circ)
        txt = ax.text(x, y, name, ha="center", va="center", fontsize=6, color="white")
        return circ, txt

    def animate(i):
        for c, t in drawn.values():
            c.remove(); t.remove()
        drawn.clear()
        for name, x, y, state in frame_list[i]:
            drawn[name] = draw_customer(name, x, y, state)
        time_text.set_text(f"Time: {frame_times[i]:.1f} min")
        return list(sum(drawn.values(), ())) + [time_text]

    anim = animation.FuncAnimation(fig, animate, frames=len(frame_list), interval=1000/fps, blit=False)
    plt.close(fig)

    # Save video
    if save_path:
        try:
            writer = FFMpegWriter(fps=fps, bitrate=1800)
            anim.save(save_path, writer=writer, dpi=120)
            print(f"✅ Animation saved to {save_path}")
        except Exception as e:
            print(f"❌ Error saving animation: {e}")

    return anim

In [ ]:
print("\n--- Running real-data replay (labels C1..Cn) ---")
real_results, real_events = run_replay_with_server_reallabels(df, num_servers=DEFAULT_SERVERS, queue_capacity=DEFAULT_QUEUE_CAPACITY, visualize=True)
print("Real replay results:")
display(pd.DataFrame([real_results]))

# Create animation file (per-server vertical queues)
anim_file = "real_simulation_lottemart_panakukang_queue.mp4"
anim = create_animation_per_server(real_events, num_servers=DEFAULT_SERVERS, title="Real Simulation (LotteMart Panakukang)", save_path=anim_file, fps=8)

# Try inline display (may fail if environment lacks HTML video support)
try:
    if anim is not None:
        display(HTML(anim.to_html5_video()))
    else:
        print("Animation object is None (no events). Check 'real_events' length:", len(real_events) if real_events is not None else 0)
except Exception as e:
    print("Inline preview not available (this is OK). Check saved file:", anim_file, " Error:", e)

print("\n✅ Done. The file saved as:", anim_file)
print("Open the file from Colab Files sidebar or download it to preview the animation.")


--- Running real-data replay (labels C1..Cn) ---
Real replay results:


,Average Utilization (%),Average Time in System (minutes),Average Waiting Time (minutes),Total Work Item Completed,SimTime_min
0,39.424319,4.002277,2.137785,161,152.283333


✅ Animation saved to real_simulation_lottemart_panakukang_queue.mp4



✅ Done. The file saved as: real_simulation_lottemart_panakukang_queue.mp4
Open the file from Colab Files sidebar or download it to preview the animation.


In [ ]:
results = []

# Experiment 1: Vary number of cashiers
exp1_name = "Vary Cashier"
scenarios_exp1 = [
    ("Default (5 kasir)", 5),
    ("4 kasir", 4),
    ("6 kasir", 6),
]
print("Running Experiment 1 (vary cashiers)...")
for label, n_servers in scenarios_exp1:
    res, events = run_replay_with_server_reallabels(df, num_servers=n_servers, queue_capacity=DEFAULT_QUEUE_CAPACITY, visualize=False)
    res['Scenario'] = label
    res['Experiment'] = exp1_name
    results.append(res)

Running Experiment 1 (vary cashiers)...


In [ ]:
# Experiment 2: Rush hour (burst arrivals)
exp2_name = "High Arrival (Rush Hour)"
# We'll create a burst dataset by compressing the interarrival intervals.
# Here: arrival times are moved closer by factor 1/burst_factor (i.e. burst_factor > 1 => arrivals faster)
burst_factor = 1.5   # kamu bisa ubah ke 2.0 atau 3.0 jika mau lebih ekstrim; default 1.5 = +50% arrival rate
print(f"Running Experiment 2 (rush hour) with burst_factor = {burst_factor}...")

# Build a burst dataset: compress relative times while preserving first arrival timestamp
base_first = df['ArrivalTime'].iloc[0]
relative = (df['ArrivalTime'] - base_first).dt.total_seconds() / 60.0   # dalam menit (float)
relative_burst = relative / burst_factor
df_burst = df.copy()
df_burst['ArrivalTime'] = base_first + pd.to_timedelta(relative_burst, unit='m')

# Recompute InterArrival_min for df_burst (useful if any function expects it)
df_burst = df_burst.sort_values('ArrivalTime').reset_index(drop=True)
df_burst['InterArrival_min'] = df_burst['ArrivalTime'].diff().dt.total_seconds().div(60).fillna(0)

# Scenarios for rush hour: baseline arrival compressed, simulate with 5 and 6 cashiers
scenarios_exp2 = [
    ("Default (5 kasir) - Rush", 5),
    ("6 kasir - Rush (mitigation)", 6),
]
for label, n_servers in scenarios_exp2:
    res, events = run_replay_with_server_reallabels(df_burst, num_servers=n_servers, queue_capacity=DEFAULT_QUEUE_CAPACITY, visualize=False)
    res['Scenario'] = label
    res['Experiment'] = exp2_name
    res['BurstFactor'] = burst_factor
    # Ensure res is a dictionary before updating
    if isinstance(res, dict):
        results.append(res)
    else: # Assuming res is a list or tuple containing a dictionary
        if len(res) > 0 and isinstance(res[0], dict):
             results.append(res[0])
        else:
             print(f"Warning: Unexpected result format for scenario '{label}'. Cannot append.")

Running Experiment 2 (rush hour) with burst_factor = 1.5...


In [ ]:
# Experiment 3: Queue Discipline (Single Queue System)
exp3_name = "Experiment 3 (Single Queue)"
print(f"Running {exp3_name}...")

def run_single_queue(num_servers=DEFAULT_SERVERS,
                     arrival_rate=arrival_rate,
                     service_rate=service_rate,
                     sim_time=None,
                     seed=RANDOM_SEED):

    random.seed(seed)
    np.random.seed(seed)
    env = simpy.Environment()
    sim = MultiServerSimulator(env, num_servers=num_servers)
    queue = simpy.Store(env)  # satu antrian global

    # Proses pelanggan
    def customer(env, name):
        arrival_time = env.now
        yield queue.put((name, arrival_time))  # masukkan pelanggan ke antrian

    # Proses kasir
    def cashier(env, idx):
        while True:
            name, arrival_time = yield queue.get()  # ambil pelanggan berikutnya dari antrian
            wait = env.now - arrival_time
            sim.wait_times.append(wait)

            service_time = np.random.exponential(1.0 / service_rate)
            sim.busy_time[idx] += service_time
            yield env.timeout(service_time)

            sim.system_times.append(env.now - arrival_time)
            sim.customers_completed += 1

    # Proses kedatangan pelanggan
    def arrival_generator(env):
        cust_id = 0
        while True:
            yield env.timeout(np.random.exponential(1.0 / arrival_rate))
            cust_id += 1
            env.process(customer(env, f"C{cust_id}"))

    # Jalankan proses kasir
    for i in range(num_servers):
        env.process(cashier(env, i))

    # Jalankan proses kedatangan pelanggan
    env.process(arrival_generator(env))

    if sim_time is None:
        sim_time = 150  # default waktu simulasi (menit)
    env.run(until=sim_time)

    # Hasil
    results_dict = {
        'Experiment': exp3_name,
        'Scenario': "Single Queue (All Cashiers Shared)",
        'Average Utilization (%)': (sum(sim.busy_time) / (sim.num_servers * env.now)) * 100 if env.now > 0 else 0.0,
        'Average Time in System (minutes)': np.mean(sim.system_times) if sim.system_times else 0.0,
        'Average Waiting Time (minutes)': np.mean(sim.wait_times) if sim.wait_times else 0.0,
        'Total Work Item Completed': sim.customers_completed,
        'SimTime_min': env.now
    }
    return results_dict

# Jalankan simulasi
exp3_result_single = run_single_queue()
results.append(exp3_result_single)

Running Experiment 3 (Single Queue)...


In [ ]:
# Aggregate results
res_df = pd.DataFrame(results)[[
    'Experiment','Scenario',
    'Average Utilization (%)','Average Time in System (minutes)',
    'Average Waiting Time (minutes)','Total Work Item Completed','SimTime_min'
]]


# Format numeric columns for readability
fmt_cols = ['Average Utilization (%)','Average Time in System (minutes)','Average Waiting Time (minutes)']
for c in fmt_cols:
    if c in res_df.columns:
        res_df[c] = res_df[c].map(lambda v: round(float(v), 4) if (not pd.isna(v)) else v)

print("\n--- Updated Experiment Results ---")
display(res_df)

# Save CSV baru
out_path = "DES_experiment_results_all.csv"
res_df.to_csv(out_path, index=False)
print(f"\n✅ All experiments saved to: {out_path}")


--- Updated Experiment Results ---


,Experiment,Scenario,Average Utilization (%),Average Time in System (minutes),Average Waiting Time (minutes),Total Work Item Completed,SimTime_min
0,Vary Cashier,Default (5 kasir),39.4243,4.0023,2.1378,161,152.283333
1,Vary Cashier,4 kasir,45.9396,5.2015,3.3234,149,152.283333
2,Vary Cashier,6 kasir,32.8536,4.0023,2.1378,161,152.283333
3,High Arrival (Rush Hour),Default (5 kasir) - Rush,30.5439,5.5768,3.6059,118,152.283333
4,High Arrival (Rush Hour),6 kasir - Rush (mitigation),26.2924,6.2210,4.2022,119,152.283333
5,Experiment 3 (Single Queue),Single Queue (All Cashiers Shared),69.9969,2.1947,0.2914,266,150.000000



✅ All experiments saved to: DES_experiment_results_all.csv
